# Multi-Scene Visualization

Demonstrates the `VizSceneHandle` API for creating multiple named scenes
under a single `Visualizer` server. Each scene is reachable at a unique URL
path and displayed inline in an iframe.

### URL scheme

- Main scene — `http://localhost:8765/`
- Scene "one" — `http://localhost:8765/one`
- Scene "two" — `http://localhost:8765/two`
- Scene "three" — `http://localhost:8765/three`

In [ ]:
# ── Imports ──────────────────────────────────────────
from pytanga.geometry import Point, Sphere
from pytanga.viz import Visualizer

print("✓ Imports ready")

In [ ]:
# ── Start the visualizer ─────────────────────────────
# In Jupyter, start() auto-detects the environment and
# does not wait for a browser (iframes connect later).
viz = Visualizer(port=8765, title="Multi-Scene Demo")
viz.start()
print(f"Server running at {viz.url}")

## Scene "one" — a single sphere

In [ ]:
one = viz.scene("one")
one.set_title("One Sphere")
one.add(Sphere(Point(0, 0, 0), 1.0), color="#ff4444", opacity=0.6)
one.flush()
print(f"Scene 'one' ready at {one.url}")

## Scene "two" — two spheres

In [ ]:
two = viz.scene("two")
two.set_title("Two Spheres")
two.add(Sphere(Point(-0.5, 0, 0), 0.8), color="#44aaff", opacity=0.6)
two.add(Sphere(Point(0.5, 0, 0), 0.8), color="#44ff44", opacity=0.6)
two.flush()
print(f"Scene 'two' ready at {two.url}")

## Scene "three" — three spheres

In [ ]:
three = viz.scene("three")
three.set_title("Three Spheres")
three.add(Sphere(Point(0, 0.5, 0), 0.6), color="#ff4444", opacity=0.6)
three.add(Sphere(Point(-0.5, -0.4, 0), 0.6), color="#44aaff", opacity=0.6)
three.add(Sphere(Point(0.5, -0.4, 0), 0.6), color="#44ff44", opacity=0.6)
three.flush()
print(f"Scene 'three' ready at {three.url}")

## List all available scenes

In [ ]:
viz.list_scenes()

## Display all three scenes side by side

Each `VizSceneHandle` has a `display(viewer_name=...)` method that embeds an
iframe pointing to the scene's unique URL with a `?viewer=` query parameter.
The `viewer_name` is sent to the server on WebSocket connect, enabling
per-viewer identification in `list_browsers()` and targeted navigation
via `navigate_to(target="viewer:...")`.

💡 **Tip:** The viewer at each URL is fully interactive — you can orbit,
pan, and zoom independently in each scene.

In [ ]:
from IPython.display import HTML

viz.display_row((one, "browser-one"),
                (two, "browser-two"),
                (three, "browser-three"))

## Navigate browsers to a specific scene

`navigate_to()` sends a navigation command to connected browsers.
Use `target="viewer:<name>"` to target a specific viewer.

In [ ]:
# Navigate browser 'browser-one' to scene "two"
viz.navigate_to("two", target="viewer:browser-one")
print("✓ Navigate command sent to 'browser-one'")

## Animation example — move a sphere in scene "one"

In [ ]:
import time
import math

viz.navigate_to("one", target="viewer:browser-one")

pt = one.add(Point(-1.5, 0, 0), color="#ffff44")
one.flush()

# Oscillate the point for 5 seconds
print("Animating point in scene 'one' …")
t_start = time.monotonic()
while (time.monotonic() - t_start) < 5.0:
    elapsed = time.monotonic() - t_start
    x = 1.5 * math.cos(elapsed * 2.0)
    y = 1.0 * math.sin(elapsed * 1.5)
    one.update_entity(pt, Point(x, y, 0))
    one.flush()
    time.sleep(1.0 / 60.0)

print("✓ Animation complete")

## List connected browsers

Each browser is identified by a server-assigned `id` and an optional
`viewer_name` set via the `?viewer=` URL parameter.

In [ ]:
browsers = viz.list_browsers()
if browsers:
    for b in browsers:
        viewer = b.get('viewer_name') or '(none)'
        print(f"  browser {b['id']} — viewer: {viewer} — scene: {b['scene']!r} — {b['remote_addr']}")
else:
    print("  (no browsers connected yet — open the viewer in a browser tab)")

## Navigate a specific viewer

Use `target="viewer:<name>"` to navigate only the browser with that viewer name.

In [ ]:
# Navigate only "browser-one" to scene "three"
viz.navigate_to("three", target="viewer:browser-one")
print("✓ Navigate command sent to viewer 'browser-one'")

## Cleanup

In [ ]:
viz.stop()
print("✓ Server stopped")